In [ ]:
import random
from collections import defaultdict
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import textwrap

# ------------------------- CONFIGURATION ----------------------------
# week_days = ["Sunday", "Monday","Tuesday", "Wednesday", "Thursday", "Friday"] # Moved to user input
# class_range = list(range(11, 12)) # Moved to user input
# sections = ['A', 'B'] # Moved to user input

# Global placeholders
school_start_time = None
school_end_time = None
break_start = None
break_duration = None
teacher_grade_limit = {}
part_time_teacher_availability = {}
subjects = {}
subject_periods_per_week = {}
friday_only_subjects = set()
week_days = []
class_range = []
sections = []

# ------------------------- UTILITY FUNCTIONS ----------------------------
def get_minutes_from_input(prompt):
    while True:
        try:
            time_input = input(prompt + " (HH:MM, 24-hour format): ").strip()
            hour, minute = map(int, time_input.split(":"))
            return hour * 60 + minute
        except:
            print("Invalid time format. Please use HH:MM")

def format_time(minutes):
    return f"{minutes // 60:02d}:{minutes % 60:02d}"

def wrap_cell(text, width=18):
    return '\n'.join(textwrap.wrap(text, width))

def generate_time_slots(start, end, durations, break_start, break_duration, day):
    slots = []
    current = start
    break_inserted = False # Flag to track if break has been inserted
    while current < end:
        # Insert break if it's time and not already inserted (and not Friday)
        if not break_inserted and current >= break_start and day != 'Friday':
            slots.append(("Break", break_start, break_start + break_duration))
            current = break_start + break_duration
            break_inserted = True
            continue

        # Check if adding a class period exceeds the end time
        if current + min(durations) > end:
            break

        for dur in sorted(durations):
            if current + dur <= end:
                slots.append(("Class", current, current + dur))
                current += dur
                break
        else:
            # If no duration fits, break the loop
            break

    # If break was not inserted because current never reached break_start, and it's not Friday, insert it at break_start
    if not break_inserted and break_start >= start and break_start < end and day != 'Friday':
         # Find where to insert the break based on start time
        insert_index = 0
        for i, (_, slot_start, _) in enumerate(slots):
            if slot_start >= break_start:
                insert_index = i
                break
            insert_index = i + 1 # If break is after all existing slots

        slots.insert(insert_index, ("Break", break_start, break_start + break_duration))


    return slots

# ------------------------- USER INPUT ----------------------------

def collect_user_settings():
    global school_start_time, school_end_time, break_start, break_duration, friday_only_subjects, week_days, class_range, sections
    print("\n\U0001F4D8 Select type (School / College):")
    user_type = input("Enter 'school' or 'college': ").strip().lower()
    is_school = user_type == 'school'

    print("\n\U0001F4C5 Enter Working Days")
    week_days_input = input("Enter working days (comma-separated, e.g., Sunday,Monday,Tuesday): ").strip()
    week_days = [day.strip() for day in week_days_input.split(',')]

    print("\n\U0001F4D8 Enter Class Ranges")
    class_range_input = input("Enter class ranges (comma-separated, e.g., 11,12 or 1-5): ").strip()
    class_range_str = [c.strip() for c in class_range_input.split(',')]
    for r in class_range_str:
        if '-' in r:
            start, end = map(int, r.split('-'))
            class_range.extend(list(range(start, end + 1)))
        else:
            class_range.append(int(r))
    class_range = sorted(list(set(class_range))) # Remove duplicates and sort

    print("\n\U0001F3EB Enter Sections")
    sections_input = input("Enter sections (comma-separated, e.g., A,B,C): ").strip()
    sections = [sec.strip() for sec in sections_input.split(',')]


    print("\n\U0001F527 Enter School/College Daily Timings")
    school_start_time = get_minutes_from_input("Start time")
    school_end_time = get_minutes_from_input("End time")
    break_start = get_minutes_from_input("Break start time")
    break_duration = int(input("Break duration in minutes: "))

    print("\n\U0001F469‍\U0001F3EB Enter Subject, Teacher and Max Grade")
    print("(Format: Subject, Teacher, MaxGrade — type 'done' to finish)")
    while True:
        inp = input("→ ").strip()
        if inp.lower() == 'done':
            break
        try:
            subject, teacher, max_grade = map(str.strip, inp.split(','))
            max_grade = int(max_grade)
            if subject not in subjects:
                subjects[subject] = []
            subjects[subject].append((teacher, 45))
            teacher_grade_limit[teacher] = max_grade
        except:
            print("Invalid format. Try: Subject, Teacher, MaxGrade")

    print("\n\U0001F9CD‍♂️ Part-time Teacher Availability")
    print("(Format: Teacher, Day, HH:MM-HH:MM — type 'done' to finish)")
    while True:
        inp = input("→ ").strip()
        if inp.lower() == 'done':
            break
        try:
            teacher, day, time_range = map(str.strip, inp.split(','))
            start_str, end_str = time_range.split('-')
            start = int(start_str.split(':')[0]) * 60 + int(start_str.split(':')[1])
            end = int(end_str.split(':')[0]) * 60 + int(end_str.split(':')[1])
            part_time_teacher_availability.setdefault(teacher, {}).setdefault(day, []).append((start, end))
        except:
            print("Invalid format. Try: Teacher, Day, HH:MM-HH:MM")

    print("\n\U0001F4D8 Enter Periods per Week per Class for each Subject")
    for cls in class_range:
        for sec in sections:
            key = f"{cls}-{sec}"
            subject_periods_per_week[key] = {}
            print(f"\nClass {key}:")
            for subject in subjects:
                while True:
                    try:
                        periods = int(input(f"Periods for {subject}: "))
                        subject_periods_per_week[key][subject] = periods
                        break
                    except:
                        print("Invalid number. Try again.")

    if is_school:
        friday_only_subjects = set(input("Enter Friday-only subjects (comma-separated): ").split(','))

    return is_school

# ------------------------- TIMETABLE GENERATION ----------------------------

def generate_routine(is_school):
    durations = [45]
    routine = defaultdict(lambda: defaultdict(list))
    teacher_schedule = defaultdict(lambda: defaultdict(list))
    teacher_assigned_classes = defaultdict(lambda: defaultdict(set))

    for cls in class_range:
        for sec in sections:
            class_sec = f"{cls}-{sec}"
            period_count = subject_periods_per_week[class_sec].copy()
            assigned_per_day_count = defaultdict(lambda: defaultdict(int))  # subjects assigned per day count
            for day in week_days:
                time_slots = generate_time_slots(school_start_time, school_end_time, durations, break_start, break_duration, day)
                prev_subject = None
                for i, (label, start, end) in enumerate(time_slots):
                    if label == "Break":
                        routine[class_sec][day].append((format_time(start), format_time(end), "Break", ""))
                        prev_subject = None
                        continue

                    subjects_left = []
                    for subj, left in period_count.items():
                        if left == 0:
                            continue
                        if subj in friday_only_subjects and day != "Friday":
                            continue
                        if assigned_per_day_count[day][subj] > 0: # Check if subject is already assigned
                            continue
                        if is_school and prev_subject == subj:
                            continue
                        if not is_school and assigned_per_day_count[day][subj] >= 2: # Check count directly
                            continue
                        subjects_left.append((subj, left))

                    if not subjects_left:
                        continue

                    random.shuffle(subjects_left)
                    scheduled = False

                    for subject, _ in subjects_left:
                        suitable_teachers = [t for t in subjects[subject] if teacher_grade_limit.get(t[0], 99) >= cls]
                        random.shuffle(suitable_teachers)
                        for teacher, _ in suitable_teachers:
                            if any(s < end and e > start for s, e in teacher_schedule[teacher][day]):
                                continue
                            if teacher in teacher_assigned_classes[class_sec][day]:
                                continue
                            if teacher in part_time_teacher_availability:
                                allowed_times = part_time_teacher_availability[teacher].get(day, [])
                                if not any(start >= a and end <= b for a, b in allowed_times):
                                    continue
                            routine[class_sec][day].append((format_time(start), format_time(end), subject, teacher))
                            teacher_schedule[teacher][day].append((start, end))
                            teacher_assigned_classes[class_sec][day].add(teacher)
                            assigned_per_day_count[day][subject] += 1  # Increment the count
                            period_count[subject] -= 1
                            prev_subject = subject
                            scheduled = True
                            break
                        if scheduled:
                            break

            if any(period_count[subj] > 0 for subj in period_count):
                print(f"\u26a0\ufe0f Could not assign all periods for {class_sec}:",
                      {subj: count for subj, count in period_count.items() if count > 0})
    return routine

# ------------------------- VIEW ROUTINE ----------------------------

def view_routine(routine):
    import textwrap

    def wrap_cell(text, width=18):
        return '\n'.join(textwrap.wrap(text, width))

    for class_sec in sorted(routine.keys()):
        table = {day: [] for day in week_days}
        max_len = 0

        for day in week_days:
            entries = []
            for start, end, subject, teacher in sorted(routine[class_sec][day]):
                raw_text = f"{start}-{end}\n{subject}\n({teacher})"
                entries.append(wrap_cell(raw_text))
            table[day] = entries
            max_len = max(max_len, len(entries))

        for day in week_days:
            table[day] += [""] * (max_len - len(table[day]))

        df = pd.DataFrame(table)

        fig_height = max(4, len(df) * 0.9)
        fig_width = max(10, len(week_days) * 2.5)

        fig, ax = plt.subplots(figsize=(fig_width, fig_height))
        ax.axis('off')

        table_plot = ax.table(
            cellText=df.values,
            colLabels=df.columns,
            cellLoc='center',
            loc='center'
        )

        table_plot.auto_set_font_size(False)
        table_plot.set_fontsize(9)
        table_plot.scale(1.3, 1.8)  # Scale for better spacing

        plt.title(f"View Timetable for Class {class_sec}", fontsize=16, fontweight='bold', pad=20)
        plt.tight_layout()
        plt.show()

# ------------------------- MAIN ----------------------------
if __name__ == "__main__":
    try:
        is_school = collect_user_settings()
        routine = generate_routine(is_school)
        view_routine(routine)
    except Exception as e:
        print("\u274c Error generating timetable:", e)


📘 Select type (School / College):
Enter 'school' or 'college': college

📅 Enter Working Days
